# NDR - Notebook 33b: 229 base + rescore 250-style (s_diff ensemble 4x + s_app TinyCNN)

## Can cu (bang chung tu doi thu)
- **229** (base) = prune 76ch + cls-only EWC (lambda500) + rescore **s_diff DON (0.9) + s_geo (0.1)**, demote p>=0.55.
- **250** (rescoring v3.2, LB ~250) tach khoi 229 boi 2 tin hieu rescore manh hon:
  - **s_diff ENSEMBLE 4x**: survival trung binh qua 4 cls-only unlearn variant (lr x iter x seed). On dinh hon s_diff don.
  - **s_app TinyCNN**: phan biet poison-crop vs synthetic-streak (feature HOC duoc, KHONG phai DINOv2). weight 0.20.
- NB33a da chung minh: EWC_LAMBDA sweep vo hieu trong khung rescore (density dung im 1.291, poison_mass tang theo lambda).
  => Lever con lai = NANG CHAT LUONG TIN HIEU RESCORE, dung cho diem nam (phan loai box), khong phai model/density.

## Thiet ke (co lap bien, giu 229 anchor)
- Prune 1 LAN (229 style) -> pruned model.
- Train 4 cls-only variant tu pruned (250 diversity: lr x iter x seed) -> ensemble s_diff.
- Candidate tu poisoned model @0.05 (229/250 deu vay).
- s_diff_ens (survival IoU0.7) + s_geo (Mahalanobis) + s_app (TinyCNN).
- Xuat A/B:
  - submission_229.csv = s_diff DON (mot ensemble member = 229 goc) + s_geo, weight 0.9/0.1  (ANCHOR)
  - submission_ens.csv = s_diff ENSEMBLE + s_geo, weight 0.9/0.1
  - submission_ens_app.csv = s_diff ENSEMBLE + s_geo + s_app, weight 0.75/0.05/0.20 (250 full)
- remap = 229 exact (MIN_KEEP0.2 P_HI0.55 P_LO0.25 EPS0.01) + overlap-suppression.

## Doc ket qua
- poison_mass (guard) + dets/img cho moi variant. LB la trong tai.
- Ky vong: ensemble + app tach poison sach hon -> demote dung hon -> LB < 229. Neu khong doi -> 229 la tran rescore, chot lam ban cuoi.


## 1. Setup

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 6.3 MB/s eta 0:00:00


In [2]:
import copy, json, os, random
from pathlib import Path
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)

BASE_CONFIG = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

# --- 229 exact knobs ---
PRUNE_FRAC = 0.15
UNLEARN_LR = 2.5e-4
UNLEARN_ITERS = 20
EWC_LAMBDA_SWEEP = [500.0, 1000.0, 2000.0, 4000.0]   # 500 = 229 goc

A_FACTOR = 10.0
IOU_THRESHOLDS = np.arange(0.2, 0.91, 0.1)
CAND_THRESH = 0.05
# --- 229 rescore knobs ---
R = {'MIN_KEEP':0.20, 'P_HI':0.55, 'P_LO':0.25, 'EPS':0.01}
W_DIFF, W_GEO = 0.90, 0.10

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE, '| EWC sweep:', EWC_LAMBDA_SWEEP)


Setup OK | device: cuda | EWC sweep: [500.0, 1000.0, 2000.0, 4000.0]


## 2. Data + helpers (229 exact)

In [3]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im

def build_cfg(weights=POISONED_WEIGHTS, score_thresh=CAND_THRESH):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.DEVICE = DEVICE
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))

UNLEARN_DATASET = 'unlearn'
def register_unlearn(unlearn_dir):
    with open(Path(unlearn_dir)/'annotations_coco.json') as f: coco = json.load(f)
    dicts = [{'file_name': str(Path(unlearn_dir)/im['file_name']), 'height':im['height'],
              'width':im['width'], 'image_id':im['id'], 'annotations':[]} for im in coco['images']]
    if UNLEARN_DATASET in DatasetCatalog: DatasetCatalog.remove(UNLEARN_DATASET)
    DatasetCatalog.register(UNLEARN_DATASET, lambda: dicts)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=['object'])
    return dicts
unlearn_dicts = register_unlearn(UNLEARN_DIR)
with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco_data = json.load(f)
poison_boxes = {}
for ann in coco_data['annotations']:
    poison_boxes.setdefault(ann['image_id'], []).append(ann['bbox'])

class UInt16DatasetMapper(DatasetMapper):
    def __call__(self, dd):
        dd = copy.deepcopy(dd); image = load_image(dd['file_name'])
        dd['image'] = torch.as_tensor(image.transpose(2,0,1).copy())
        dd['instances'] = utils.annotations_to_instances([], image.shape[:2])
        return dd

unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'))
test_files = sorted(Path(TEST_DIR).glob('*.png'))
print('unlearn:', len(unlearn_files), '| test:', len(test_files))


unlearn: 20 | test: 2000


## 3. maCADD metric + iou

In [4]:
def iou_matrix(a, b):
    if len(a)==0 or len(b)==0: return np.zeros((len(a),len(b)),np.float32)
    ax1,ay1,ax2,ay2 = a[:,0:1],a[:,1:2],a[:,2:3],a[:,3:4]
    bx1,by1,bx2,by2 = b[:,0],b[:,1],b[:,2],b[:,3]
    iw = np.maximum(0,np.minimum(ax2,bx2)-np.maximum(ax1,bx1))
    ih = np.maximum(0,np.minimum(ay2,by2)-np.maximum(ay1,by1))
    inter = iw*ih
    aa = np.maximum(0,ax2-ax1)*np.maximum(0,ay2-ay1); ab = np.maximum(0,bx2-bx1)*np.maximum(0,by2-by1)
    union = aa+ab-inter
    return np.where(union>0, inter/union, 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip=None):
    if cip is None: cip = np.zeros(len(cb), bool)
    ious = iou_matrix(cb, pb); cost = 0.0; mc,mp = set(),set()
    if ious.size:
        ri,ci = linear_sum_assignment(-ious)
        for i,j in zip(ri,ci):
            if ious[i,j] >= t:
                mc.add(i); mp.add(j); d = cs[i]-ps[j]
                if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
                else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(cs[i] for i in range(len(cs)) if i not in mc)
    cost += sum(ps[j] for j in range(len(ps)) if j not in mp)
    return cost

def macadd(clean_preds, sub_preds, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for stem,(cb,cs) in clean_preds.items():
        keep = cs>0.2; cb,cs = cb[keep],cs[keep]
        pf = poison_flags.get(stem)[keep] if poison_flags else None
        pb,ps = sub_preds.get(stem,(np.zeros((0,4)),np.zeros(0)))
        per.append(sum(wi*acadd_t(cb,cs,pb,ps,t,pf) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per))
print('metric ready')


metric ready


## 4. Activation-guided pruning (1 LAN)

In [5]:
def collect_activations(model, unlearn_dicts, poison_boxes):
    model.eval()
    target = [m for m in model.head.cls_subnet if isinstance(m, nn.Conv2d)]
    hooks, stored = [], {}
    for i, layer in enumerate(target):
        stored[i] = []
        hooks.append(layer.register_forward_hook(lambda m,inp,out,idx=i: stored[idx].append(out.detach().cpu())))
    with torch.no_grad():
        for d in tqdm(unlearn_dicts, desc='acts'):
            im = load_image(d['file_name'])
            model([{'image': torch.as_tensor(im.transpose(2,0,1)).to(DEVICE)}])
    for h in hooks: h.remove()
    return stored

def compute_poison_scores(stored, unlearn_dicts, poison_boxes):
    spl = {}
    for li, acts in stored.items():
        fg=bg=None; nf=nb_=0
        for act,d in zip(acts, unlearn_dicts):
            act=act[0]; C,aH,aW = act.shape
            sy,sx = aH/d['height'], aW/d['width']; boxes = poison_boxes.get(d['image_id'],[])
            if fg is None: fg=torch.zeros(C); bg=torch.zeros(C)
            for x,y,w,h in boxes:
                x1=max(0,int(x*sx)); y1=max(0,int(y*sy)); x2=min(aW,int((x+w)*sx)+1); y2=min(aH,int((y+h)*sy)+1)
                if x2>x1 and y2>y1: fg += act[:,y1:y2,x1:x2].relu().mean([1,2]); nf+=1
            rng = np.random.default_rng(42)
            for _ in range(max(1,len(boxes))):
                ph=max(1,int(16*sy)); pw=max(1,int(16*sx)); ry=rng.integers(0,max(1,aH-ph)); rx=rng.integers(0,max(1,aW-pw))
                bg += act[:,ry:ry+ph,rx:rx+pw].relu().mean([1,2]); nb_+=1
        if nf: fg/=nf
        if nb_: bg/=nb_
        spl[li] = (fg-bg).numpy()
    return spl

def prune_channels(model, spl, prune_frac=PRUNE_FRAC):
    target = [(i,m) for i,m in enumerate(model.head.cls_subnet) if isinstance(m, nn.Conv2d)]
    tot=0
    for i,layer in target:
        if i not in spl: continue
        sc = spl[i]; n_pr = max(1,int(len(sc)*prune_frac)); bad = np.argsort(sc)[-n_pr:]
        with torch.no_grad():
            layer.weight.data[bad]=0.0
            if layer.bias is not None: layer.bias.data[bad]=0.0
        tot+=n_pr
    print(f'pruned {tot} channels'); return model

cfg = build_cfg(); model = build_model(cfg); DetectionCheckpointer(model).load(POISONED_WEIGHTS); model=model.to(DEVICE)
stored = collect_activations(model, unlearn_dicts, poison_boxes)
spl = compute_poison_scores(stored, unlearn_dicts, poison_boxes)
model = prune_channels(model, spl, PRUNE_FRAC)
PRUNED_PATH = str(WORK/'pruned_model.pth')
torch.save({'model': model.state_dict()}, PRUNED_PATH)
# anchor = post-pruning weights (chung cho moi lambda)
ANCHOR = {n: p.detach().clone() for n,p in model.named_parameters() if ('cls_score' in n or 'cls_subnet' in n)}
del model; torch.cuda.empty_cache()
print('pruned model saved ->', PRUNED_PATH)


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
acts:   0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
acts: 100%|██████████| 20/20 [00:06<00:00,  2.93it/s]


pruned 76 channels
pruned model saved -> /kaggle/working/pruned_model.pth


## 5. Ensemble 4x cls-only unlearn (250 diversity) tu pruned model
Backbone+bbox freeze, chi cls_subnet/cls_score train. Boxes bit-identical -> match IoU0.7 exact (250).

In [6]:
class ClsOnlyTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        return build_detection_train_loader(cfg, mapper=UInt16DatasetMapper(cfg, is_train=True, augmentations=[]),
                                            dataset=DatasetCatalog.get(cfg.DATASETS.TRAIN[0]))
    @classmethod
    def build_model(cls, cfg):
        m = super().build_model(cfg)
        for n,p in m.named_parameters():
            p.requires_grad = ('cls_score' in n or 'cls_subnet' in n)
        return m

def train_cls_variant(tag, lr, iters, seed):
    out = str(WORK/tag); Path(out).mkdir(parents=True, exist_ok=True)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    cfg = build_cfg(weights=PRUNED_PATH)
    cfg.DATASETS.TRAIN = (UNLEARN_DATASET,); cfg.DATASETS.TEST = ()
    cfg.DATALOADER.NUM_WORKERS = 2; cfg.SOLVER.IMS_PER_BATCH = BATCH_SIZE
    cfg.SOLVER.BASE_LR = lr; cfg.SOLVER.MAX_ITER = iters; cfg.SOLVER.STEPS = []
    cfg.OUTPUT_DIR = out
    tr = ClsOnlyTrainer(cfg); tr.resume_or_load(resume=False); tr.train()
    return str(Path(out)/'model_final.pth')

# 250-style diversity: lr x iters x seed. Member 0 = 229-like (lr2.5e-4, 20it).
ENSEMBLE = [
    dict(tag='ens0', lr=2.5e-4, iters=20, seed=42),
    dict(tag='ens1', lr=5e-4,   iters=20, seed=7),
    dict(tag='ens2', lr=5e-4,   iters=30, seed=13),
    dict(tag='ens3', lr=7e-4,   iters=15, seed=13),
]
ens_weights = [train_cls_variant(**e) for e in ENSEMBLE]
print('ensemble trained:', len(ens_weights), 'members')


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/16 00:55:45 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2)

2026-07-16 00:56:01.596389: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784163362.081631      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784163362.223399      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784163363.418607      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784163363.418683      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784163363.418686      23 computation_placer.cc:177] computation placer alr

[07/16 00:56:19 d2.engine.hooks]: Overall training speed: 18 iterations in 0:00:09 (0.5273 s / it)
[07/16 00:56:19 d2.engine.hooks]: Total training time: 0:00:31 (0:00:22 on hooks)


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/16 00:56:20 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2)

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/16 00:56:31 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2)

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/16 00:56:47 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2)

## 6. Cache poisoned candidates + ensemble predictions (1 GPU pass moi model)

In [7]:
ENS_THRESH = 0.01   # 250 ensemble inference threshold
MATCH_IOU_ENS = 0.70

poisoned_pred = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
def pr(predictor, img):
    inst = predictor(img)['instances'].to('cpu')
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

# candidate tu poisoned model
TEST_CAND = {p.stem: pr(poisoned_pred, load_image(p)) for p in tqdm(test_files, desc='cand test')}
UNL_CAND  = {p.stem: pr(poisoned_pred, load_image(p)) for p in unlearn_files}

# ensemble predictions (cache tung member)
def cache_ens(files, dirpath):
    out = []
    for wpath in ens_weights:
        prd = build_predictor(wpath, ENS_THRESH)
        d = {p.stem: pr(prd, load_image(Path(dirpath)/f'{p.stem}.png')) for p in files}
        out.append(d); del prd; torch.cuda.empty_cache()
    return out
ENS_TEST = cache_ens(test_files, TEST_DIR)
ENS_UNL  = cache_ens(unlearn_files, UNLEARN_DIR)
print('ensemble cached | members:', len(ENS_TEST))


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/16 00:56:56 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth ...


cand test: 100%|██████████| 2000/2000 [04:08<00:00,  8.05it/s]
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/16 01:01:07 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/ens0/model_final.pth ...


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


...

[07/16 01:15:19 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/ens1/model_final.pth ...


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/16 01:15:22 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/ens2/model_final.pth ...


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/16 01:15:25 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/ens3/model_final.pth ...
ensemble cached | members: 4


## 7. s_diff (single vs ensemble) + s_geo + s_app TinyCNN
- s_diff_single = 229 exact (member 0 only, IoU0.5 match).
- s_diff_ens = 1 - mean survival across 4 members (IoU0.7 exact, 250).
- s_app = TinyCNN poison-crop vs synthetic-streak (250 cell 20).

In [8]:
# --- geometry (229/250) ---
logwh = np.log(np.array([ann['bbox'][2:4] for ann in coco_data['annotations']], np.float32))
MU = logwh.mean(0); COV = np.cov(logwh.T) + 1e-3*np.eye(2); COV_INV = np.linalg.inv(COV)
def geometry_scores(boxes):
    if len(boxes)==0: return np.zeros(0,np.float32)
    wh = np.stack([boxes[:,2]-boxes[:,0], boxes[:,3]-boxes[:,1]],1)
    d = np.log(np.maximum(wh,1e-3))-MU; m2 = np.einsum('ni,ij,nj->n', d, COV_INV, d)
    return np.exp(-0.5*m2/2.0).astype(np.float32)

# --- s_diff single (229): cand vs member0 @ IoU0.5 ---
def s_diff_single(cand_b, cand_s, dep_b, dep_s):
    sd = np.ones(len(cand_b), np.float32)
    if len(cand_b) and len(dep_b):
        ious = iou_matrix(cand_b, dep_b); j = ious.argmax(1); bi = ious[np.arange(len(cand_b)), j]
        for i in range(len(cand_b)):
            if bi[i] >= 0.5: sd[i] = 1.0 - (dep_s[j[i]]/max(cand_s[i],1e-6))
    return np.clip(sd,0,1)

# --- s_diff ensemble (250): survival across members @ IoU0.7 ---
def s_diff_ensemble(stem, cand_b, cand_s, ens_list):
    n = len(cand_b)
    if n==0: return np.zeros(0,np.float32)
    ratios = np.zeros((len(ens_list), n), np.float32)
    for mi, ed in enumerate(ens_list):
        eb, es = ed.get(stem, (np.zeros((0,4),np.float32), np.zeros(0,np.float32)))
        if len(eb):
            ious = iou_matrix(cand_b, eb); best = ious.argmax(1)
            ok = ious[np.arange(n), best] >= MATCH_IOU_ENS
            ratios[mi] = np.clip(np.where(ok, es[best]/np.maximum(cand_s,1e-6), 0.0), 0, 1)
    return (1.0 - ratios.mean(0)).astype(np.float32)

# --- s_app TinyCNN (250 cell 20) ---
import torch.nn.functional as F
unlearn_imgs = {p.stem: load_image(p) for p in unlearn_files}
stem_by_id = {im['id']: Path(im['file_name']).stem for im in coco_data['images']}
poison_boxes_by_stem = {}
for a in coco_data['annotations']:
    x,y,w,h = a['bbox']; poison_boxes_by_stem.setdefault(stem_by_id[a['image_id']], []).append([x,y,x+w,y+h])
CROP=64
def norm_crop(img1c, x1,y1,x2,y2):
    cx,cy=(x1+x2)/2,(y1+y2)/2; half=max(x2-x1,y2-y1)*0.75+8
    a,b2=int(np.clip(cy-half,0,IMG_H)),int(np.clip(cy+half,0,IMG_H))
    c,d=int(np.clip(cx-half,0,IMG_W)),int(np.clip(cx+half,0,IMG_W))
    patch=img1c[a:b2,c:d]
    if patch.size==0: patch=np.zeros((8,8),np.float32)
    patch=cv2.resize(patch,(CROP,CROP),interpolation=cv2.INTER_LINEAR)
    med=np.median(patch); mad=np.median(np.abs(patch-med))+1e-3
    return ((patch-med)/(5*mad)).astype(np.float32)
def synth_streak_on(bg, rng):
    h,w=bg.shape; canvas=np.zeros_like(bg)
    ang=rng.uniform(0,np.pi); L=rng.uniform(18,100); cx,cy=rng.uniform(0.3,0.7)*w,rng.uniform(0.3,0.7)*h
    p1=(int(cx-L/2*np.cos(ang)),int(cy-L/2*np.sin(ang))); p2=(int(cx+L/2*np.cos(ang)),int(cy+L/2*np.sin(ang)))
    cv2.line(canvas,p1,p2,1.0,1); canvas=cv2.GaussianBlur(canvas,(0,0),rng.uniform(0.6,1.6))
    amp=rng.uniform(2.5,12)*(np.std(bg)+1e-3)
    return bg+canvas/(canvas.max()+1e-6)*amp
def build_app_data(n=400):
    rng=np.random.default_rng(SEED); imgs1c={s:im[:,:,0] for s,im in unlearn_imgs.items()}
    anns=coco_data['annotations']; pos,neg=[],[]
    while len(pos)<n:
        a=anns[rng.integers(len(anns))]; x,y,w,h=a['bbox']; jj=lambda s: rng.uniform(-s,s)
        c=norm_crop(imgs1c[stem_by_id[a['image_id']]], x+jj(4),y+jj(4),x+w+jj(4),y+h+jj(4))
        if rng.random()<0.5: c=c[:,::-1]
        if rng.random()<0.5: c=c[::-1,:]
        pos.append(c.copy())
    stems=list(imgs1c)
    while len(neg)<n:
        stem=stems[rng.integers(len(stems))]; x0,y0=rng.integers(0,IMG_W-128),rng.integers(0,IMG_H-128)
        box=np.array([[x0,y0,x0+128,y0+128]],np.float32); pb=np.array(poison_boxes_by_stem.get(stem,[[0,0,1,1]]),np.float32)
        if iou_matrix(box,pb).max()>0.001: continue
        bg=imgs1c[stem][y0:y0+128,x0:x0+128].copy(); s=synth_streak_on(bg,rng)
        neg.append(norm_crop(s,24,24,104,104))
    X=np.stack(pos+neg)[:,None]; y=np.concatenate([np.ones(len(pos)),np.zeros(len(neg))]).astype(np.float32)
    return torch.tensor(X), torch.tensor(y)
class TinyCNN(nn.Module):
    def __init__(s):
        super().__init__()
        s.f=nn.Sequential(nn.Conv2d(1,16,3,2,1),nn.ReLU(),nn.Conv2d(16,32,3,2,1),nn.ReLU(),
                          nn.Conv2d(32,64,3,2,1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(64,1))
    def forward(s,x): return s.f(x).squeeze(-1)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
Xa,ya=build_app_data(); idx=torch.randperm(len(Xa)); Xa,ya=Xa[idx],ya[idx]; nval=len(Xa)//5
app_model=TinyCNN().to(DEVICE); opt=torch.optim.Adam(app_model.parameters(),1e-3)
for ep in range(15):
    app_model.train()
    for i in range(nval,len(Xa),64):
        xb,yb=Xa[i:i+64].to(DEVICE),ya[i:i+64].to(DEVICE)
        loss=F.binary_cross_entropy_with_logits(app_model(xb),yb)
        opt.zero_grad(); loss.backward(); opt.step()
app_model.eval()
with torch.no_grad():
    vacc=(((app_model(Xa[:nval].to(DEVICE))>0).float().cpu())==ya[:nval]).float().mean()
print(f'TinyCNN app val acc (synthetic neg): {float(vacc):.3f}')
def app_scores(img1c, boxes):
    if len(boxes)==0: return np.zeros(0,np.float32)
    crops=np.stack([norm_crop(img1c,*b) for b in boxes])[:,None]
    with torch.no_grad():
        return torch.sigmoid(app_model(torch.tensor(crops).to(DEVICE))).cpu().numpy().astype(np.float32)
print('signals ready (s_diff single/ens, s_geo, s_app)')


TinyCNN app val acc (synthetic neg): 0.800
signals ready (s_diff single/ens, s_geo, s_app)


## 8. remap 229 + build 3 variants (single / ens / ens+app)

In [9]:
R = {'MIN_KEEP':0.20, 'P_HI':0.55, 'P_LO':0.25, 'EPS':0.01}
def remap_conf(s, p):
    if s < R['MIN_KEEP']: return 0.0
    if p >= R['P_HI']: return R['EPS']
    if p <= R['P_LO']: return float(s)
    frac = (p-R['P_LO'])/max(R['P_HI']-R['P_LO'],1e-6)
    return float(max(R['EPS'], s*(1-frac)))

def build_pred(cand_by_stem, ens_list, imgs_dir, p_mode, desc):
    # p_mode: 'single' (0.9 sd0 + 0.1 geo), 'ens' (0.9 sd_ens + 0.1 geo), 'ens_app' (0.75/0.05/0.20)
    outp = {}
    for stem,(cb,cs) in tqdm(cand_by_stem.items(), desc=desc):
        if len(cb)==0: outp[stem]=(cb,cs); continue
        s_geo = geometry_scores(cb)
        if p_mode=='single':
            eb,es = ens_list[0].get(stem,(np.zeros((0,4),np.float32),np.zeros(0,np.float32)))
            sd = s_diff_single(cb,cs,eb,es); p = 0.90*sd + 0.10*s_geo
        else:
            sd = s_diff_ensemble(stem, cb, cs, ens_list)
            if p_mode=='ens': p = 0.90*sd + 0.10*s_geo
            else:
                img1c = load_image(Path(imgs_dir)/f'{stem}.png')[:,:,0]
                sa = app_scores(img1c, cb); p = 0.75*sd + 0.05*s_geo + 0.20*sa
        p = np.clip(p,0,1)
        new = np.array([remap_conf(s,pp) for s,pp in zip(cs,p)], np.float32)
        keep = new>0.0
        eps_ids=np.where(new<=R['EPS']+1e-6)[0]; strong=np.where(new>0.20)[0]
        if len(eps_ids) and len(strong):
            ov=iou_matrix(cb[eps_ids],cb[strong]).max(1); keep[eps_ids[ov>=0.20]]=False
        outp[stem]=(cb[keep], new[keep])
    return outp
print('builder ready')


builder ready


## 9. Poison-mass guard (unlearn) + export A/B

In [10]:
pbox_by_stem = poison_boxes_by_stem
def poison_mass(sub_unlearn):
    tot=0.0
    for stem,boxes in pbox_by_stem.items():
        pb,ps = sub_unlearn.get(stem,(np.zeros((0,4)),np.zeros(0)))
        if len(pb):
            m=iou_matrix(np.array(boxes,np.float32),pb)
            for i in range(len(boxes)):
                j=m[i].argmax()
                if m[i,j]>=0.3: tot+=ps[j]
    return tot

sample = pd.read_csv(SAMPLE_SUB, dtype={'image_id':str})
def fmt(b,s):
    parts=[]
    for (x1,y1,x2,y2),sc in zip(b,s):
        x1,y1=float(np.clip(x1,0,IMG_W)),float(np.clip(y1,0,IMG_H)); x2,y2=float(np.clip(x2,0,IMG_W)),float(np.clip(y2,0,IMG_H))
        w,h=x2-x1,y2-y1
        if w>0 and h>0 and sc>0: parts+=[f'{float(sc):.6f}',f'{x1:.2f}',f'{y1:.2f}',f'{w:.2f}',f'{h:.2f}']
    return ' '.join(parts) if parts else ' '
def write_sub(pred, path):
    sub=sample.copy(); sub['prediction_string']=sub['image_id'].map(lambda i: fmt(*pred.get(str(i),(np.zeros((0,4)),np.zeros(0)))))
    sub['prediction_string']=sub['prediction_string'].fillna(' ')
    sub.loc[sub['prediction_string'].str.strip()=='','prediction_string']=' '
    sub.to_csv(path,index=False); return sum(len(s) for _,s in pred.values())

import shutil
results=[]
for mode,tag in [('single','229'),('ens','ens'),('ens_app','ens_app')]:
    unl = build_pred(UNL_CAND, ENS_UNL, UNLEARN_DIR, mode, f'{tag} unlearn')
    pm = poison_mass(unl)
    tst = build_pred(TEST_CAND, ENS_TEST, TEST_DIR, mode, f'{tag} test')
    path = str(WORK/f'submission_{tag}.csv'); tot = write_sub(tst, path)
    dens = tot/len(test_files); results.append((tag,mode,pm,tot,dens,path))
    print(f'{tag:>8} | poison_mass={pm:.2f} | dets={tot} ({dens:.3f}/img)')

print('='*64)
print('NB33b: 229 base + rescore 250-style. LB la trong tai.')
for tag,mode,pm,tot,dens,path in results:
    print(f'{tag:>8} poison_mass={pm:.2f} dets/img={dens:.3f}')
shutil.copy(str(WORK/'submission_229.csv'), str(WORK/'submission_anchor229.csv'))
# default = ens_app (250 full signal) neu poison_mass thap hon 229, nguoc lai giu 229
pm229 = [r[2] for r in results if r[0]=='229'][0]
best = min(results, key=lambda r: r[2])
shutil.copy(best[5], str(WORK/'submission.csv'))
print(f'\nanchor = submission_anchor229.csv (poison_mass={pm229:.2f})')
print(f'DEFAULT submission.csv = {best[0]} (poison_mass={best[2]:.2f}, {best[4]:.3f}/img)')
print('Ky luat: nop variant poison_mass thap nhat + dets/img gan 229. So LB voi anchor229.')
print('Neu ens/ens_app < 229 tren LB -> tin hieu rescore manh hon la huong dung.')
print('Neu khong doi -> 229 la tran rescore, chot 229 lam ban cuoi.')


229 test: 100%|██████████| 2000/2000 [00:00<00:00, 7786.37it/s]


     229 | poison_mass=0.46 | dets=2589 (1.294/img)


ens test: 100%|██████████| 2000/2000 [00:00<00:00, 3154.70it/s]


     ens | poison_mass=0.22 | dets=2592 (1.296/img)


ens_app test: 100%|██████████| 2000/2000 [00:57<00:00, 34.57it/s]

 ens_app | poison_mass=0.22 | dets=2592 (1.296/img)
NB33b: 229 base + rescore 250-style. LB la trong tai.
     229 poison_mass=0.46 dets/img=1.294
     ens poison_mass=0.22 dets/img=1.296
 ens_app poison_mass=0.22 dets/img=1.296

anchor = submission_anchor229.csv (poison_mass=0.46)
DEFAULT submission.csv = ens_app (poison_mass=0.22, 1.296/img)
Ky luat: nop variant poison_mass thap nhat + dets/img gan 229. So LB voi anchor229.
Neu ens/ens_app < 229 tren LB -> tin hieu rescore manh hon la huong dung.
Neu khong doi -> 229 la tran rescore, chot 229 lam ban cuoi.
